In [ ]:
from figure_paths import DATA, RESULTS, WORK, FIGURES, setting
from pathlib import Path
import json
import warnings
import importlib.util
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator, FuncFormatter, ScalarFormatter

DATA_DIR = RESULTS
OUTPUT_DIR = DATA_DIR
LEADS = (1, 2, 3, 4, 5, 6)
COMPONENTS = ("u", "v", "w")
PLOT_COMPONENTS = ("u", "v", "w")  # w 文件对应训练时使用的 -0.1 × ERA5 omega
PLOT_LEVELS_HPA = (850, 925, 1000)
BASELINE_SUFFIX = ""
RESIDUAL_SUFFIX = "_residual"
COMMON_TIME_ACROSS_LEADS = True  # 六个预报时效也使用相同的目标时次
TIME_START = None               # 默认使用文件内完整的共同测试时段
TIME_END = None                 # 如需复现旧的截短时段，可设 "2023-10-26 23:00:00"
TIME_ROUND_TOLERANCE_SECONDS = 1.0
COORD_ATOL = 1e-6
CHUNK_SIZE = 128                # 按时间分块读取，不将多年风场一次性加载进内存
MIN_VALID_TIMES = 3
TRUTH_RTOL, TRUTH_ATOL = 1e-6, 1e-7
FONT_SCALE = 0.8
FIGSIZE = (12.6, 6.9)
SAVE_DPI = 600
USE_CARTOPY = True              # 未安装或无可用海岸线数据时，自动使用经纬度坐标图
METRICS_PATH = OUTPUT_DIR / "da_transunet_residual_vs_baseline_spatial_metrics_1to6h.nc"

for _old_ds in globals().get("_opened_datasets", []):
    _old_ds.close()
_opened_datasets = []

def input_path(lead, component, experiment, kind):
    suffix = BASELINE_SUFFIX if experiment == "baseline" else RESIDUAL_SUFFIX
    return DATA_DIR / f"da_transunet_{kind}_{component}_{lead}_24year_changjiang{suffix}.nc"

def canonical_field(path, variable):
    ds = xr.open_dataset(path, cache=False)
    _opened_datasets.append(ds)
    if variable not in ds:
        raise KeyError(f"{path.name}: 未找到变量 {variable}，现有变量 {list(ds.data_vars)}")
    da = ds[variable]
    aliases = {"lat": "latitude", "lon": "longitude", "plev": "level",
               "pressure_level": "level", "valid_time": "time"}
    da = da.rename({old: new for old, new in aliases.items()
                    if old in da.dims and new not in da.dims})
    required = ("time", "level", "latitude", "longitude")
    if set(da.dims) != set(required):
        raise ValueError(f"{path.name}: 需要维度 {required}，实际 {da.dims}")
    da = da.transpose(*required)
    if not np.issubdtype(da.time.dtype, np.datetime64):
        raise TypeError(f"{path.name}: time 未解码为 datetime64，请检查 NetCDF 时间单位/日历。")
    times = pd.DatetimeIndex(da.time.values)
    if times.hasnans:
        raise ValueError(f"{path.name}: 存在 NaT。")
    rounded = times.round("h")
    deviations = np.abs((times - rounded).total_seconds())
    if np.any(deviations > TIME_ROUND_TOLERANCE_SECONDS):
        raise ValueError(f"{path.name}: 存在偏离整点超过容限的时间，最大 {deviations.max():.6g} 秒。")
    if rounded.has_duplicates:
        raise ValueError(f"{path.name}: 时间取整后出现重复值，不能自动配对。")
    da = da.assign_coords(time=rounded.to_numpy()).sortby("time")
    level_attrs = dict(da.level.attrs)
    levels = np.asarray(da.level.values, dtype=float)
    if str(level_attrs.get("units", "")).strip().lower() == "pa" or np.nanmax(levels) > 2000:
        levels = levels / 100.0
    da = da.assign_coords(level=levels)
    da.level.attrs.update(units="hPa")
    for dim in ("level", "latitude", "longitude"):
        vals = np.asarray(da[dim].values, dtype=float)
        if not np.all(np.isfinite(vals)) or len(np.unique(vals)) != len(vals):
            raise ValueError(f"{path.name}: {dim} 坐标含缺失或重复值。")
        da = da.sortby(dim)
    return da.sel(time=slice(TIME_START, TIME_END))

_specs = [(h, c, e, k) for h in LEADS for c in COMPONENTS
          for e in ("baseline", "residual") for k in ("testy", "predicty")]
_missing = [str(input_path(*s)) for s in _specs if not input_path(*s).is_file()]
if _missing:
    raise FileNotFoundError("以下文件不存在，请检查 DATA_DIR 或文件名：\n" + "\n".join(_missing))
if not set(PLOT_COMPONENTS).issubset(COMPONENTS) or not PLOT_COMPONENTS:
    raise ValueError("PLOT_COMPONENTS 必须是 COMPONENTS 的非空子集。")

input_data, manifest, alignment_rows = {}, {}, []
try:
    _reference = None
    for h, c, e, k in _specs:
        path = input_path(h, c, e, k)
        da = canonical_field(path, f"{k}_{c}")
        if not da.sizes["time"]:
            raise ValueError(f"{path.name}: 指定时间范围内无数据。")
        if _reference is None:
            _reference = da
        for dim in ("level", "latitude", "longitude"):
            a, b = np.asarray(da[dim].values), np.asarray(_reference[dim].values)
            if a.shape != b.shape or not np.allclose(a, b, rtol=0, atol=COORD_ATOL):
                raise ValueError(f"{path.name}: {dim} 网格不一致，不进行隐式插值。")
            da = da.assign_coords({dim: b})
        input_data[(h, c, e, k)] = da
        manifest[f"{h}h/{c}/{e}/{k}"] = str(path)
    _available_levels = np.asarray(_reference.level.values, dtype=float)
    _selected_levels = []
    for p in PLOT_LEVELS_HPA:
        match = _available_levels[np.isclose(_available_levels, p, rtol=0, atol=COORD_ATOL)]
        if len(match) != 1:
            raise ValueError(f"绘图压力层 {p} hPa 不在数据中：{_available_levels}")
        _selected_levels.append(float(match[0]))
    if not _selected_levels or len(set(_selected_levels)) != len(_selected_levels):
        raise ValueError("PLOT_LEVELS_HPA 不能为空或包含重复压力层。")
    if len(set(PLOT_COMPONENTS)) != len(PLOT_COMPONENTS):
        raise ValueError("PLOT_COMPONENTS 不能重复。")
    PLOT_LEVELS_HPA = tuple(_selected_levels)

    common_times = {}
    for h in LEADS:
        arrays = [da for key, da in input_data.items() if key[0] == h]
        times = pd.DatetimeIndex(arrays[0].time.values)
        for da in arrays[1:]:
            times = times.intersection(pd.DatetimeIndex(da.time.values))
        common_times[h] = times.sort_values()
    if COMMON_TIME_ACROSS_LEADS:
        times = common_times[LEADS[0]]
        for h in LEADS[1:]:
            times = times.intersection(common_times[h])
        common_times = {h: times.sort_values() for h in LEADS}
    for h in LEADS:
        times = common_times[h]
        if len(times) < MIN_VALID_TIMES:
            raise ValueError(f"{h}h: 共同测试时次不足。")
        for key in [key for key in input_data if key[0] == h]:
            before = input_data[key].sizes["time"]
            input_data[key] = input_data[key].sel(time=times.to_numpy())
            alignment_rows.append({"lead_time": h, "component": key[1], "experiment": key[2],
                                   "kind": key[3], "original_count": before,
                                   "aligned_count": len(times), "dropped_count": before-len(times)})
        print(f"{h}h: {len(times):,} 个共同目标时次；{times[0]} 至 {times[-1]}")
    alignment_report = pd.DataFrame(alignment_rows)
    print(f"已读取 {len(input_data)} 个文件的坐标信息。按时间分块计算；不使用 GPU。")
    print("各文件因对齐而剔除的时次数：", sorted(alignment_report.dropped_count.unique().tolist()))
except Exception:
    for _ds in _opened_datasets:
        _ds.close()
    _opened_datasets = []
    raise


In [ ]:
def paired_temporal_metrics(base_truth, base_pred, residual_truth, residual_pred,
                            chunk_size=128, min_count=3, truth_rtol=1e-6, truth_atol=1e-7):
    """R 为时间 Pearson r；RMSPE = RMSE / (各模型自身 truth 的时间极差)。

    数据须已按 time/level/latitude/longitude 精确对齐。使用分块中心矩合并，
    避免大偏置数据在 sum(x*x)-sum(x)**2/n 中产生的消减误差。
    不同实验的 truth 不互相替代；差异只进入审计输出，不阻止计算。
    """
    arrays = xr.align(base_truth, base_pred, residual_truth, residual_pred, join="exact", copy=False)
    arrays = tuple(a.transpose("time", "level", "latitude", "longitude") for a in arrays)
    if chunk_size < 1 or min_count < 2:
        raise ValueError("chunk_size >= 1，min_count >= 2。")
    shape = arrays[0].shape[1:]
    n = np.zeros(shape, dtype=np.int64)
    means = np.zeros((4,) + shape, dtype=np.float64)
    m2 = np.zeros_like(means)
    covariance = np.zeros((2,) + shape, dtype=np.float64)
    squared_error = np.zeros_like(covariance)
    truth_min = np.full_like(covariance, np.inf)
    truth_max = np.full_like(covariance, -np.inf)
    truth_pairs = np.zeros(shape, dtype=np.int64)
    truth_mismatch = np.zeros(shape, dtype=np.int64)
    truth_max_diff = np.zeros(shape, dtype=np.float64)

    for start in range(0, arrays[0].sizes["time"], chunk_size):
        stop = min(start + chunk_size, arrays[0].sizes["time"])
        block = np.stack([np.asarray(a.isel(time=slice(start, stop)).values, dtype=np.float64)
                          for a in arrays], axis=0)
        valid = np.all(np.isfinite(block), axis=0)
        nb = valid.sum(axis=0)
        safe_nb = np.maximum(nb, 1)
        batch_means = np.where(valid[None, ...], block, 0.0).sum(axis=1) / safe_nb
        centered = np.where(valid[None, ...], block - batch_means[:, None, ...], 0.0)
        batch_m2 = np.sum(centered * centered, axis=1)
        batch_cov = np.stack([(centered[i] * centered[i+1]).sum(axis=0) for i in (0, 2)])
        total = n + nb
        safe_total = np.maximum(total, 1)
        delta = batch_means - means
        factor = n * (nb / safe_total)
        m2 += batch_m2 + delta * delta * factor
        covariance += batch_cov + np.stack([delta[i] * delta[i+1] * factor for i in (0, 2)])
        means += delta * (nb / safe_total)
        n = total
        for j, i in enumerate((0, 2)):
            error = np.where(valid, block[i+1] - block[i], 0.0)
            squared_error[j] += np.sum(error * error, axis=0)
            truth_min[j] = np.minimum(truth_min[j], np.where(valid, block[i], np.inf).min(axis=0))
            truth_max[j] = np.maximum(truth_max[j], np.where(valid, block[i], -np.inf).max(axis=0))
        truth_finite = np.isfinite(block[0]) & np.isfinite(block[2])
        difference = np.abs(block[0] - block[2])
        truth_pairs += truth_finite.sum(axis=0)
        truth_mismatch += (truth_finite & (difference > truth_atol + truth_rtol*np.abs(block[0]))).sum(axis=0)
        truth_max_diff = np.maximum(truth_max_diff, np.where(truth_finite, difference, 0.0).max(axis=0))

    result = {"n_valid": n, "truth_pair_count": truth_pairs,
              "truth_mismatch_count": truth_mismatch,
              "truth_max_abs_difference": np.where(truth_pairs > 0, truth_max_diff, np.nan)}
    for j, (experiment, i) in enumerate((("baseline", 0), ("residual", 2))):
        denominator = np.sqrt(m2[i]) * np.sqrt(m2[i+1])
        r = np.full(shape, np.nan)
        np.divide(covariance[j], denominator, out=r, where=(n >= min_count) & (denominator > 0))
        rmse = np.sqrt(squared_error[j] / np.maximum(n, 1))
        data_range = truth_max[j] - truth_min[j]
        rmspe = np.full(shape, np.nan)
        np.divide(rmse, data_range, out=rmspe, where=(n >= min_count) & (data_range > 0))
        result[f"r_{experiment}"] = np.clip(r, -1.0, 1.0)
        result[f"rmspe_{experiment}"] = rmspe
    return result


In [ ]:
_records = []
try:
    for h in LEADS:
        _component_records = []
        for c in COMPONENTS:
            values = paired_temporal_metrics(
                input_data[(h, c, "baseline", "testy")], input_data[(h, c, "baseline", "predicty")],
                input_data[(h, c, "residual", "testy")], input_data[(h, c, "residual", "predicty")],
                chunk_size=CHUNK_SIZE, min_count=MIN_VALID_TIMES,
                truth_rtol=TRUTH_RTOL, truth_atol=TRUTH_ATOL)
            source = input_data[(h, c, "baseline", "testy")]
            coords = {d: source[d].values for d in ("level", "latitude", "longitude")}
            ds_part = xr.Dataset({name: (("level", "latitude", "longitude"), value)
                                  for name, value in values.items()}, coords=coords)
            _component_records.append(ds_part.expand_dims(component=[c]))
            count = int(values["truth_mismatch_count"].sum())
            if count:
                max_diff = float(np.nanmax(values["truth_max_abs_difference"]))
                warnings.warn(f"{h}h/{c}: 发现 {count:,} 个 truth 数值配对超出容限；"
                              f"最大绝对差={max_diff:.6g}（文件原单位）。"
                              "已使用各自 truth；差异已记入 nc，比较解释需考虑目标差异。")
            print(f"完成 {h}h / {c}")
        _records.append(xr.concat(_component_records, dim="component").expand_dims(lead_time=[h]))
    metrics_ds = xr.concat(_records, dim="lead_time")
finally:
    for _ds in _opened_datasets:
        _ds.close()
    _opened_datasets = []

for name in ("r_baseline", "r_residual", "rmspe_baseline", "rmspe_residual"):
    metrics_ds[name].attrs.update(units="1", temporal_dimension="time",
                                  evaluation_mask="finite baseline truth/prediction AND residual truth/prediction")
metrics_ds["delta_r"] = metrics_ds.r_residual - metrics_ds.r_baseline
metrics_ds["delta_rmspe"] = metrics_ds.rmspe_baseline - metrics_ds.rmspe_residual
_reduce_dims = ("component", "level")
_selected_count = len(PLOT_COMPONENTS) * len(PLOT_LEVELS_HPA)
for name in ("delta_r", "delta_rmspe"):
    selected = metrics_ds[name].sel(component=list(PLOT_COMPONENTS), level=list(PLOT_LEVELS_HPA))
    finite = np.isfinite(selected)
    metrics_ds[f"{name}_contribution_count"] = finite.sum(dim=_reduce_dims).astype("int16")
    metrics_ds[f"{name}_map"] = selected.where(finite).mean(dim=_reduce_dims, skipna=False)
    metrics_ds[f"{name}_map"].attrs.update(units="1", positive_means="residual model improves",
        aggregation="equal mean of temporal metric differences over selected components and pressure levels",
        required_contribution_count=_selected_count)
metrics_ds.delta_r.attrs["definition"] = "R_residual - R_baseline"
metrics_ds.delta_rmspe.attrs["definition"] = "RMSPE_baseline - RMSPE_residual"
metrics_ds["aligned_time_count"] = ("lead_time", [len(common_times[h]) for h in LEADS])
metrics_ds["evaluation_start"] = ("lead_time", np.array([common_times[h][0].to_datetime64() for h in LEADS]))
metrics_ds["evaluation_end"] = ("lead_time", np.array([common_times[h][-1].to_datetime64() for h in LEADS]))
metrics_ds.lead_time.attrs.update(units="hours", long_name="forecast lead time")
metrics_ds.level.attrs.update(units="hPa")
metrics_ds.latitude.attrs.update(units="degrees_north")
metrics_ds.longitude.attrs.update(units="degrees_east")
metrics_ds.attrs.update(
    title="Residual-corrected versus single-stage DA-TransUNet: temporal metrics and spatial differences",
    rmspe_definition="sqrt(mean((prediction-truth)**2)) / (max(truth)-min(truth)); no multiplication by 100",
    truth_policy="Each model uses its own saved testy; shared timestamps and four-array finite mask.",
    common_time_across_leads=int(COMMON_TIME_ACROSS_LEADS),
    selected_components=json.dumps(list(PLOT_COMPONENTS)), selected_levels_hpa=json.dumps(list(PLOT_LEVELS_HPA)),
    plot_mask="All selected component/level metrics must be finite for a map grid point.",
    truth_rtol=TRUTH_RTOL, truth_atol=TRUTH_ATOL,
    source_files=json.dumps(manifest, ensure_ascii=False),
    time_alignment=json.dumps(alignment_rows, ensure_ascii=False),
    created_utc=pd.Timestamp.now(tz="UTC").isoformat())
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_engine = "netcdf4" if importlib.util.find_spec("netCDF4") is not None else "scipy"
_encoding = {name: {"zlib": True, "complevel": 4} for name in metrics_ds.data_vars} if _engine == "netcdf4" else {}
_temporary = METRICS_PATH.with_name(METRICS_PATH.stem + ".tmp.nc")
try:
    metrics_ds.to_netcdf(_temporary, engine=_engine, encoding=_encoding)
    _temporary.replace(METRICS_PATH)
finally:
    if _temporary.exists():
        _temporary.unlink()
print(f"指标已保存：{METRICS_PATH}")
print("以下为空间等权均值（并非将所有时空样本合并后的指标）：")
for h in LEADS:
    dr = metrics_ds.delta_r_map.sel(lead_time=h).values
    de = metrics_ds.delta_rmspe_map.sel(lead_time=h).values
    mr = float(np.nanmean(dr)) if np.any(np.isfinite(dr)) else np.nan
    me = float(np.nanmean(de)) if np.any(np.isfinite(de)) else np.nan
    print(f"{h}h: mean ΔR={mr:+.6f}，mean ΔRMSPE={me:+.6f}；"
          f"有效格点 R={np.isfinite(dr).sum()}，RMSPE={np.isfinite(de).sum()}")
